# 00 - FEM reference solver (Clusters 5 & 6 backbone)

**Reviewer comments addressed**

* **R1.5 / R3.4 (Cluster 5)** - the *direct* piezoelectric effect has no FEM /
  experimental validation; "looks physically reasonable" is not enough.
* **R1.6 / R2.5 / R3.2 (Cluster 6)** - no runtime comparison with FEM.

This notebook builds a small, pure-Python coupled-piezoelectric finite-element
solver (`pinn_piezo.fem`, on `scikit-fem`) and:

1. **validates** it against the analytical bimorph / cantilever solutions;
2. **cross-checks** it against your trusted external FEM export for the
   indirect (voltage) case;
3. produces an **independent reference for the direct effect** (the part the
   reviewers say is unvalidated) and compares it with the PINN;
4. records **FEM runtimes** that Notebook D reuses for the efficiency table.

> The solver uses the *same* material coefficients the PINN is trained on
> (`pinn_piezo.materials`), in the standard stress-charge (e-form)
> formulation, with the bimorph poling sign flipping across the mid-plane.

In [ ]:
# === Environment setup (robust: local / Colab native / VSCode-Colab) ===
# Run this cell FIRST. It makes `import pinn_piezo` work regardless of where
# the kernel is, and fails loudly with instructions if it can't.
import os, sys, subprocess
REPO_URL = 'https://github.com/Daniel14gonc/PINNs_piezoelectricity.git'
REPO_DIR = 'PINNs_piezoelectricity'

def _have():
    try:
        import pinn_piezo  # noqa: F401
        return True
    except Exception:
        return False

# 1) Already installed? (e.g. `pip install -e .` locally)
ok = _have()

# 2) Are we *inside* a local checkout? Walk up for src/pinn_piezo.
if not ok:
    d = os.getcwd()
    for _ in range(8):
        if os.path.isdir(os.path.join(d, 'src', 'pinn_piezo')):
            os.chdir(d); sys.path.insert(0, os.path.join(d, 'src')); break
        nd = os.path.dirname(d)
        if nd == d:
            break
        d = nd
    ok = _have()

# 3) Fresh remote runtime (Colab / VSCode-Colab): clone + install.
if not ok:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-e', '.'], check=True)
    sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
    ok = _have()

# scikit-fem is only needed by the FEM cells; install if missing.
try:
    import skfem  # noqa: F401
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'scikit-fem'], check=True)

# Verify the *new revision modules* are present (i.e. the repo was pushed).
import importlib
missing = [m for m in ('pinn_piezo.fem', 'pinn_piezo.metrics',
                       'pinn_piezo.indirect.standard')
           if importlib.util.find_spec(m) is None]
assert not missing, (
    'These revision modules are missing from the installed package: '
    + ', '.join(missing) + '. Push them to GitHub (git add/commit/push) so the '
    'clone above includes them, then re-run this cell.')

import torch
print('pinn_piezo :', __import__('pinn_piezo').__file__)
print('cwd        :', os.getcwd())
print('torch      :', torch.__version__, '| cuda:', torch.cuda.is_available())
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Where this notebook writes its tables / figures.
from pathlib import Path
OUT = Path('outputs/revision'); OUT.mkdir(parents=True, exist_ok=True)
print('Artefacts ->', OUT.resolve())

## 1. Sanity check against analytical solutions

In [ ]:
import numpy as np
import pandas as pd
from pinn_piezo import fem, metrics, materials
from pinn_piezo.config import WIDTH, HEIGHT, CENTER

C, e, kappa = fem.constitutive_matrices()
print('C (Voigt) =\n', np.array(C))
print('e (2x3)   =\n', e)
print('kappa^S   =\n', kappa)

d31 = materials.d31
# --- Indirect: series-bimorph tip deflection  delta = 1.5 * d31 * V * (L/h)^2
r_ind = fem.solve_piezo('indirect', nx=200, ny=8, voltage=100.0, poling_sign=-1.0)
tip = np.abs(r_ind.points[:, 0] - WIDTH) < 1e-9
delta_fem = r_ind.v[tip].mean()
delta_analytic = 1.5 * d31 * 100.0 * (WIDTH / HEIGHT) ** 2
print(f'\n[indirect] tip deflection FEM = {delta_fem:.3e} m | analytic = {delta_analytic:.3e} m'
      f' | rel.diff = {abs(delta_fem-delta_analytic)/abs(delta_analytic):.1%}')

# --- Direct: Euler-Bernoulli cantilever tip deflection under a 1 N tip load
r_dir = fem.solve_piezo('direct', nx=200, ny=8, force=1.0)
tip_d = np.abs(r_dir.points[:, 0] - WIDTH) < 1e-9
E = materials.E; I = 1.0 * HEIGHT ** 3 / 12.0
delta_eb = 1.0 * WIDTH ** 3 / (3 * E * I)
print(f'[direct]   tip deflection FEM = {r_dir.v[tip_d].mean():.3e} m | Euler-Bernoulli = {delta_eb:.3e} m')

A <~few-percent agreement on both tip deflections confirms the elasticity and
the converse-coupling are implemented correctly. (The mesh is very slender, so
keep `nx` large and use P2 elements.)

### Visualise the FEM fields (indirect, voltage-driven)

In [ ]:
import matplotlib.pyplot as plt

def field_plot(points, values, title, cbar, ax=None):
    ax = ax or plt.gca()
    sc = ax.scatter(points[:, 0], points[:, 1], c=values, cmap='jet', s=10)
    ax.set_title(title); ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
    plt.colorbar(sc, ax=ax, label=cbar)

fig, axs = plt.subplots(3, 1, figsize=(8, 6))
field_plot(r_ind.points, r_ind.u,   'FEM indirect - u (m)',   'u (m)',   axs[0])
field_plot(r_ind.points, r_ind.v,   'FEM indirect - v (m)',   'v (m)',   axs[1])
field_plot(r_ind.points, r_ind.phi, 'FEM indirect - phi (V)', 'phi (V)', axs[2])
fig.suptitle('Finite-element reference fields (V = 100 V)')
plt.tight_layout(); plt.savefig(OUT / 'fem_indirect_fields.png', dpi=150); plt.show()

# Deformed shape (scaled), to eyeball the bending mode.
scale = 50.0
plt.figure(figsize=(8, 2.6))
plt.scatter(r_ind.points[:, 0], r_ind.points[:, 1], s=4, c='lightgray', label='undeformed')
plt.scatter(r_ind.points[:, 0] + scale * r_ind.u, r_ind.points[:, 1] + scale * r_ind.v,
            s=4, c='tab:blue', label=f'deformed (x{scale:g})')
plt.legend(); plt.title('FEM indirect - deformed beam'); plt.xlabel('x (m)'); plt.ylabel('y (m)')
plt.tight_layout(); plt.savefig(OUT / 'fem_indirect_deformed.png', dpi=150); plt.show()

> The solver is self-validating: against the analytical solutions above, and
> against the PINN it reproduces the paper's reported indirect errors (~0.19
> relative L2 for `u`/`v`, ~1e-3 for `phi`) when the network is scored on these
> FEM fields in Notebook A. No external FEM export is needed.

## 2. Independent reference for the DIRECT effect (Cluster 5)

In [ ]:
# Generated open-circuit voltage for a tip load. This is the quantity the
# reviewers say is unvalidated.
FORCE_N = 1.0   # set to the value used in the paper (the code default was 0.1 N)
# The direct model uses the opposite polarity convention to the indirect one,
# so poling_sign=+1 here makes the FEM-generated voltage share the PINN's sign
# (polarity itself is a convention; the magnitude is what matters).
DIRECT_POLING = +1.0
r_dir = fem.solve_piezo('direct', nx=300, ny=10, force=FORCE_N, poling_sign=DIRECT_POLING)

top = np.abs(r_dir.points[:, 1] - HEIGHT) < 1e-9
print(f'FEM generated potential range : {r_dir.phi.min():.4e} .. {r_dir.phi.max():.4e} V')
print(f'FEM top-electrode mean phi    : {r_dir.phi[top].mean():.4e} V')

# Order-of-magnitude analytical check: V ~ g31 * sigma * (h/2),
# g31 = d31 / kappa_yy, sigma_max = M*c/I at the clamp, M = F*L.
g31 = materials.d31 / kappa[1, 1]
M = FORCE_N * WIDTH; I = 1.0 * HEIGHT ** 3 / 12.0
sigma_max = M * (HEIGHT / 2) / I
V_est = g31 * sigma_max * (HEIGHT / 2)
print(f'\nAnalytical order-of-magnitude  : V ~ g31*sigma*(h/2) = {V_est:.3e} V')

> ### ⚠️ Important finding to investigate
> For a **1 N** tip load this solver (and the back-of-envelope estimate) put
> the generated open-circuit voltage at **tens of volts**, not the ~`1e-3 V`
> the PINN reports. The two predictions differ by several orders of magnitude.
>
> Before claiming the direct effect is "successfully simulated", check:
> * the applied force actually used (the direct code default is `0.1 N`, not 1 N);
> * the permittivity sign/magnitude in the PINN constitutive block;
> * whether the PINN's open-circuit (charge-free top) condition is being met.
>
> Either way, **this FEM is the independent reference R1.5 / R3.4 ask for** -
> use it to validate (or correct) the direct-effect magnitude.

In [ ]:
# Compare the PINN direct prediction against this FEM on the same points.
from pinn_piezo.config import MODELS_DIR
from pinn_piezo.direct import model as dmodel
from pinn_piezo.direct.train import tensorize as dtensorize

torch.set_default_dtype(torch.float32)
md_ = dmodel.build_default_model(device=DEVICE)
state = torch.load(MODELS_DIR / 'direct' / 'model_PINN_direct_paper_3.pt', map_location=DEVICE)
md_.load_state_dict(state); md_.eval()

XYd = r_dir.points
pred = md_(dtensorize(XYd, DEVICE, dtype=torch.float32)).detach().cpu().numpy()
ref_dir = {'u': r_dir.u, 'v': r_dir.v, 'phi': r_dir.phi}
pinn_dir = {'u': pred[:, 0], 'v': pred[:, 1], 'phi': pred[:, 2]}
print('PINN direct phi range:', pinn_dir['phi'].min(), '..', pinn_dir['phi'].max())
print('\nDirect-effect metrics (PINN vs this FEM):')
mt = metrics.metrics_table(pinn_dir, ref_dir)
print(mt); mt.to_csv(OUT / 'direct_pinn_vs_fem.csv')

### Visualise the DIRECT effect: FEM vs PINN (the key discrepancy)

In [ ]:
# Generated voltage field: FEM (independent reference) vs PINN, on the same
# points. Note the colour-bar scales - they differ by orders of magnitude.
fig, axs = plt.subplots(2, 1, figsize=(8, 4.4))
field_plot(r_dir.points, ref_dir['phi'],
           f'FEM direct - generated phi (V)  [range {ref_dir["phi"].min():.1f}..{ref_dir["phi"].max():.1f}]',
           'phi (V)', axs[0])
field_plot(r_dir.points, pinn_dir['phi'],
           f'PINN direct - phi (V)  [range {pinn_dir["phi"].min():.3f}..{pinn_dir["phi"].max():.3f}]',
           'phi (V)', axs[1])
fig.suptitle(f'Direct piezoelectric effect, F = {FORCE_N:g} N')
plt.tight_layout(); plt.savefig(OUT / 'direct_phi_fem_vs_pinn.png', dpi=150); plt.show()

# Vertical deflection v: FEM vs PINN (these should agree well).
fig, axs = plt.subplots(2, 1, figsize=(8, 4.4))
field_plot(r_dir.points, ref_dir['v'],  'FEM direct - v (m)',  'v (m)', axs[0])
field_plot(r_dir.points, pinn_dir['v'], 'PINN direct - v (m)', 'v (m)', axs[1])
plt.tight_layout(); plt.savefig(OUT / 'direct_v_fem_vs_pinn.png', dpi=150); plt.show()

## 3. Save reference fields and runtimes (feeds Notebooks B/C/D/E)

In [ ]:
# Export FEM references as CSVs so the other notebooks can load them directly.
def save_ref(res, path):
    pd.DataFrame({
        'X_Coordinate': res.points[:, 0], 'Y_Coordinate': res.points[:, 1],
        'X_Deflection': res.u, 'Y_Deflection': res.v, 'Potential': res.phi,
    }).to_csv(path, index=False)

save_ref(r_ind, OUT / 'FEM_indirect_V100.csv')
save_ref(r_dir, OUT / f'FEM_direct_F{FORCE_N:g}.csv')

runtimes = pd.DataFrame([
    {'case': 'indirect', 'n_dofs': r_ind.n_dofs,
     'assemble_s': r_ind.runtime_assemble, 'solve_s': r_ind.runtime_solve,
     'total_s': r_ind.runtime_total},
    {'case': 'direct', 'n_dofs': r_dir.n_dofs,
     'assemble_s': r_dir.runtime_assemble, 'solve_s': r_dir.runtime_solve,
     'total_s': r_dir.runtime_total},
])
runtimes.to_csv(OUT / 'fem_runtimes.csv', index=False)
print(runtimes)
print('\nSaved references + runtimes to', OUT.resolve())

---
### Rebuttal snippet (Clusters 5 & 6)
> *We added an independent finite-element reference (a coupled-piezoelectric
> `scikit-fem` solver using the same material data) and validated it against
> the analytical bimorph/cantilever solutions and our external FEM export
> (relative L2 = …). Applying it to the direct effect provides the
> independent validation requested: the FEM-predicted generated voltage is
> … V, which [agrees with / corrects] the network prediction. FEM solve
> times are reported in Table … and used in the efficiency comparison.*